# RetainIQ - Predictive Modeling, Survival Analysis & Interpretability
**Author:** Nitesh | Final Year Mini Project, AI & DS II

### Workflow Overview:
1. **Churn Classification:** Benchmark Logistic Regression baseline vs XGBoost with imbalance weighting (`scale_pos_weight`).
2. **Model Evaluation:** F1-score, Precision, Recall, ROC-AUC, Confusion Matrix, and ROC Curve.
3. **Explainable AI (SHAP):** TreeExplainer summary and feature impact attribution.
4. **Survival Analysis:** Kaplan-Meier survival trajectories across customer segments and Cox Proportional Hazards modeling.
5. **Future LTV Prediction:** XGBoost Regressor for forward 90-day spend with MAE and RMSE metrics.

In [1]:
import sys
sys.path.append('..')
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from src.train import train_and_save_all_models
from src.evaluate import evaluate_all_models

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Train Models & Pipelines
We execute the full training workflow across classification, regression, and survival analysis.

In [2]:
test_data = train_and_save_all_models()
metrics = evaluate_all_models()
print(json.dumps(metrics, indent=2))

### Written Insight: Churn Model Comparison
- **XGBoost Superiority:** XGBoost outperforms Logistic Regression with **F1 = 0.7559**, **Recall = 0.7513**, and **ROC-AUC = 0.7854** (vs LR F1 = 0.7340, Recall = 0.6955).
- **Class Imbalance Handling:** Setting `scale_pos_weight` directly in XGBoost counterbalances the 57.3% churn class ratio, preventing the model from disproportionately favoring false negatives in customer departure detection.

## 2. Global Interpretability via SHAP
SHAP values measure the marginal contribution of each RFM attribute to predicted churn risk.

In [3]:
xgb_churn = joblib.load('../models/xgb_churn_model.joblib')
explainer = joblib.load('../models/shap_explainer.joblib')
shap_values = explainer(test_data['X_test'])

plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, test_data['X_test'], show=False)
plt.title('SHAP Feature Attribution: XGBoost Churn Model', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### Written Insight: Key Churn Drivers
- **Recency is Paramount:** Higher recency (inactive for >60 days) drastically drives up log-odds of churn.
- **Frequency and Basket Diversity:** Customers purchasing frequently and exploring higher unique product counts display high retention resilience.
- **Tenure Dynamic:** Long-standing accounts exhibit moderate loyalty buffers, but high recency quickly degrades tenure advantages.

## 3. Survival Curves (Kaplan-Meier Analysis)

In [4]:
kmf_dict = joblib.load('../models/kmf_segments.joblib')
plt.figure(figsize=(9, 5))
for label, kmf in kmf_dict.items():
    kmf.plot_survival_function(linewidth=2.2 if label != 'Global' else 3)
plt.title('Kaplan-Meier Trajectories by Customer Segment', fontweight='bold')
plt.xlabel('Days After Cutoff')
plt.ylabel('Survival Probability')
plt.ylim(0, 1.05)
plt.show()

### Written Insight: Survival Trajectories
- **Champions** maintain the highest survival profile throughout the 90 days.
- **Hibernating and At Risk** accounts show steep early drop-offs, indicating that if interventions are not applied within the first 14-30 days, churn is nearly guaranteed.